In [ ]:
import os
import re
import time
import torch
import random
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from abctoolkit.utils import Barline_regexPattern
from abctoolkit.duration import calculate_bartext_duration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
QWEN_MODEL_ID          = "Qwen/Qwen2.5-3B-Instruct"
QWEN_MOTIF_ADAPTER_PATH = "/usr/xtmp/cy232/accompgen/distil/qwen_motif"  # final fine-tuned adapter

# ── Generation settings ────────────────────────────────────────────────────────
INFER_TOP_K         = 50
INFER_TOP_P         = 0.9
INFER_TEMPERATURE   = 1.2
INFER_MAX_NEW_TOKENS = 4096

SYSTEM_PROMPT = (
    "You are a music composition assistant that generates polyphonic ABC notation scores. "
    "Given a header describing period, composer, instrumentation and optionally a motivic pattern, "
    "generate the complete tunebody in ABC notation."
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# ── Load model ─────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

if os.path.isdir(QWEN_MOTIF_ADAPTER_PATH):
    model = PeftModel.from_pretrained(base_model, QWEN_MOTIF_ADAPTER_PATH)
    print(f"Loaded LoRA adapter from {QWEN_MOTIF_ADAPTER_PATH}")
else:
    model = base_model
    print("Warning: adapter path not found, using base model.")

model.eval()
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── Post-processing helpers (unchanged from the original demo) ─────────────────

def complete_brackets(s):
    stack = []
    bracket_map = {'{': '}', '[': ']', '(': ')'}
    for char in s:
        if char in bracket_map:
            stack.append(char)
        elif char in bracket_map.values():
            for key, value in bracket_map.items():
                if value == char:
                    if stack and stack[-1] == key:
                        stack.pop()
                    break
    return s + ''.join(bracket_map[c] for c in reversed(stack))


def rest_unreduce(abc_lines):
    tunebody_index = None
    for i in range(len(abc_lines)):
        if abc_lines[i].startswith('%%score'):
            abc_lines[i] = complete_brackets(abc_lines[i])
        if '[V:' in abc_lines[i]:
            tunebody_index = i
            break

    metadata_lines  = abc_lines[:tunebody_index]
    tunebody_lines  = abc_lines[tunebody_index:]

    part_symbol_list = []
    voice_group_list = []
    for line in metadata_lines:
        if line.startswith('%%score'):
            for m in re.findall(r'\((.*?)\)', line):
                voice_group_list.append(m.split())
            existed_voices = [v for grp in voice_group_list for v in grp]
        if line.startswith('V:'):
            sym = line.split()[0]
            part_symbol_list.append(sym)
            if sym[2:] not in existed_voices:
                voice_group_list.append([sym[2:]])

    z_symbol_list = ['V:' + grp[0] for grp in voice_group_list]
    x_symbol_list = ['V:' + v for grp in voice_group_list for v in grp[1:]]
    part_symbol_list.sort(key=lambda x: int(x[2:]))

    unreduced_lines = []
    ref_dur = None
    for i, line in enumerate(tunebody_lines):
        line = re.sub(r'^\[r:[^\]]*\]', '', line)
        matches = re.findall(r'\[V:(\d+)\](.*?)(?=\[V:|$)', line)
        line_bar_dict = {f'V:{m[0]}': m[1] for m in matches}

        dur_dict = {}
        for sym, bartext in line_bar_dict.items():
            right_barline = ''.join(re.split(Barline_regexPattern, bartext)[-2:])
            inner = bartext[:-len(right_barline)]
            try:
                d = calculate_bartext_duration(inner)
            except:
                d = None
            if d is not None:
                dur_dict[d] = dur_dict.get(d, 0) + 1
        if dur_dict:
            ref_dur = max(dur_dict, key=dur_dict.get)

        prefix = line.split('[V:')[0] if i == 0 else ''
        unreduced_line = ''
        for sym in part_symbol_list:
            if sym in line_bar_dict:
                unreduced_line += f'[{sym}]{line_bar_dict[sym]}'
            elif sym in z_symbol_list and ref_dur:
                unreduced_line += f'[{sym}]{prefix}z{ref_dur}{right_barline}'
            elif sym in x_symbol_list and ref_dur:
                unreduced_line += f'[{sym}]{prefix}x{ref_dur}{right_barline}'
        unreduced_lines.append(unreduced_line + '\n')

    return metadata_lines + unreduced_lines

In [ ]:
# ── Inference ──────────────────────────────────────────────────────────────────

def build_prompt(period, composer, instrumentation, additional_prompts=None):
    """Build the metadata header string passed to Qwen as the user turn."""
    lines = [
        f"%{period}",
        f"%{composer}",
        f"%{instrumentation}",
    ]
    if additional_prompts:
        lines += [p.rstrip('\n') for p in additional_prompts]
    return '\n'.join(lines)


def inference_qwen(period, composer, instrumentation, additional_prompts=None):
    """Generate an ABC score using the distilled Qwen model.

    Returns the post-processed ABC string, or None on failure.
    """
    header = build_prompt(period, composer, instrumentation, additional_prompts)
    print(header)

    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": header},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(device)

    with torch.inference_mode():
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            output_ids = model.generate(
                **inputs,
                max_new_tokens=INFER_MAX_NEW_TOKENS,
                do_sample=True,
                top_k=INFER_TOP_K,
                top_p=INFER_TOP_P,
                temperature=INFER_TEMPERATURE,
                pad_token_id=tokenizer.eos_token_id,
            )

    generated = tokenizer.decode(
        output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    print(generated)

    # Reconstruct full ABC: header comment lines + generated tunebody
    abc_text = header + '\n' + generated

    # Post-process
    abc_lines = [l + '\n' for l in abc_text.split('\n') if l.strip()]
    try:
        abc_lines = rest_unreduce(abc_lines)
    except Exception:
        pass

    abc_lines = [l for l in abc_lines if not (l.startswith('%') and not l.startswith('%%'))]
    abc_lines = ['X:1\n'] + abc_lines
    return ''.join(abc_lines)


def sanitize(name):
    return re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_')

## Generate with motif conditioning

In [ ]:
try:
    period          = 'Romantic'
    composer        = 'Schubert, Franz'
    instrumentation = 'Art Song'

    motif = "%motif:v1:step_skip_leap: 0,3,-1,-1 ; 0,-2,1,1 ; 0,-1,-1,-1\n"

    result = inference_qwen(period, composer, instrumentation, [motif])

    filename = f"{sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}.abc"
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(result)
    get_ipython().system(f'python ../notebook/abc2xml.py -o . {filename}')

except Exception as e:
    print(f"Error: {e}")

## Generate without motif (header only)

In [ ]:
try:
    period          = 'Classical'
    composer        = 'Beethoven, Ludwig van'
    instrumentation = 'Art Song'
    default_note, tempo, meter, key = '1/8', '1/4=80', '3/4', 'D'
    score = '1 { ( 2 4 ) | ( 3 5 ) }'

    result = inference_qwen(
        period, composer, instrumentation,
        [
            f"%%score {score} \n",
            f"L:{default_note}\n",
            f"Q:{tempo}\n",
            f"M:{meter}\n",
            f"K:{key}\n",
        ]
    )

    filename = f"{sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}.abc"
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(result)
    get_ipython().system(f'python ../notebook/abc2xml.py -o . {filename}')

except Exception as e:
    print(f"Error: {e}")

## Batch generation

In [ ]:
configs = [
    ('Classical', 'Haydn, Joseph',           'Chamber'),
    ('Classical', 'Mozart, Wolfgang Amadeus', 'Art Song'),
    ('Romantic',  'Schubert, Franz',          'Art Song'),
    ('Baroque',   'Vivaldi, Antonio',         'Orchestral'),
]

for period, composer, instrumentation in configs:
    try:
        result   = inference_qwen(period, composer, instrumentation)
        filename = f"{sanitize(period)}_{sanitize(composer)}_{sanitize(instrumentation)}.abc"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(result)
        get_ipython().system(f'python ../notebook/abc2xml.py -o . {filename}')
        print(f"Saved {filename}")
    except Exception as e:
        print(f"Failed {composer}: {e}")